# CONSTELLATION: Spatial LR Analysis of Human Lymph Node

This notebook demonstrates the full CONSTELLATION pipeline on the **10x Xenium human lymph node** dataset.

**Dataset**: 708K cells, 4,624 genes (Xenium panel), hierarchical cell type annotation  
**Method**: CONSTELLATION uses a spatial-conditioned analytical test with kernel-weighted neighborhoods to detect ligand–receptor interactions that are spatially enriched beyond expectation.

## Outline
1. Data loading & spatial overview
2. LR database & spatial graph
3. Input validation & testable burden scan
4. Run CONSTELLATION analysis
5. Results overview & cell-type pair visualizations
6. Biological validation (Tfh → GC B cell interactions)
7. LR dot plot & spatial interaction plots
8. Spatial range characterization ($\tau$ sweep)

---
## 1. Setup & Data Loading

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import constellation as cst
from scipy.spatial import cKDTree
import warnings
warnings.filterwarnings('ignore')

print(f'CONSTELLATION version: {cst.__version__}')

In [ ]:
# Load annotated Xenium lymph node data
# (produced by tutorial_annotate_lymphnode.ipynb)
DATA_PATH = '../results/xenium_lymphnode_annotated.h5ad'
adata = sc.read_h5ad(DATA_PATH)
print(f'Total cells: {adata.n_obs:,}')
print(f'Total genes: {adata.n_vars:,}')

# Filter unassigned cells
cell_type_col = 'cell_type'
adata = adata[adata.obs[cell_type_col] != 'Unassigned'].copy()
print(f'Assigned cells: {adata.n_obs:,}')

# Reverse log1p — CONSTELLATION applies log1p internally,
# so input should be normalized counts (not log-transformed)
from scipy.sparse import issparse
if issparse(adata.X):
    adata.X.data = np.expm1(adata.X.data)
else:
    adata.X = np.expm1(adata.X)
print(f'\nReversed log1p (expm1) for raw count input to CONSTELLATION')

# Cell type distribution
ct_counts = adata.obs[cell_type_col].value_counts()
print(f'\nCell types ({len(ct_counts)}):')
for ct, n in ct_counts.items():
    print(f'  {ct}: {n:,}')

### Spatial overview

The lymph node is organized into functional zones: **germinal centers** (GC) with dark zone (DZ) and light zone (LZ) B cells, **T cell zones** with naive and helper T cells, and **B cell follicle mantles** surrounding the GCs. Follicular dendritic cells (FDC) and T follicular helper (Tfh) cells are critical for GC reactions.

In [ ]:
from collections import Counter
from scipy.ndimage import gaussian_filter
from skimage import measure

# Region of interest (1 mm x 1 mm)
spatial = adata.obsm['spatial']
x_all, y_all = spatial[:, 0], spatial[:, 1]
ct_all = adata.obs[cell_type_col].values

cx, cy, sz = 4471, 5883, 1000
mask_region = ((x_all >= cx - sz/2) & (x_all <= cx + sz/2) &
               (y_all >= cy - sz/2) & (y_all <= cy + sz/2))
x_r = x_all[mask_region] - (cx - sz/2)
y_r = y_all[mask_region] - (cy - sz/2)
ct_r = ct_all[mask_region]
region_counts = Counter(ct_r)

# Cell type groups and color palettes
b_cells = ['GC_DZ', 'GC_LZ', 'B_mature', 'B_activated', 'MBC', 'NBC', 'PB', 'PC']
t_cells = ['Tfh', 'T_naive', 'T_helper', 'T_CD8', 'Treg', 'T_DN', 'NK']
stromal_myeloid = ['Endothelial', 'FDC', 'FRC', 'Stromal',
                   'Macrophage', 'Monocyte', 'Myeloid']

b_palette = ['#6a5acd', '#8b4789', '#4169a1', '#9370ab', '#7b3f7b', '#5f7daf', '#4a7c8c', '#a0628a']
t_palette = ['#cd7f32', '#a0522d', '#daa06d', '#8b4513', '#b03030', '#d2691e', '#bc8f5f']
sm_palette = ['#3c7a5a', '#7c7c3c', '#6b8e6b', '#2f5f4f', '#5a7a5a', '#5f8a8f', '#8fbc8f']

# Assign colors by abundance rank
CELL_COLORS = {}
for group, palette in [(b_cells, b_palette), (t_cells, t_palette), (stromal_myeloid, sm_palette)]:
    sorted_group = sorted([c for c in group if region_counts.get(c, 0) > 0],
                          key=lambda c: -region_counts.get(c, 0))
    for i, ct in enumerate(sorted_group):
        CELL_COLORS[ct] = palette[i] if i < len(palette) else '#888888'

def get_contours(xc, yc, sz, grid_size=40, threshold=0.25):
    bins = np.arange(0, sz + grid_size, grid_size)
    density, _, _ = np.histogram2d(xc, yc, bins=[bins, bins])
    density = gaussian_filter(density.T, sigma=1.5)
    if density.max() > 0:
        density /= density.max()
    return [c * grid_size for c in measure.find_contours(density, threshold) if len(c) > 10]

def display_name(ct):
    return ct.replace('_', ' ')

# 4-panel figure
import matplotlib.patches as mpatches
fig, axes = plt.subplots(1, 4, figsize=(20, 5.5))

# Panel 1: All cell types with zone contours
ax = axes[0]
ax.set_facecolor('white')
for ct in stromal_myeloid + b_cells + t_cells:
    m = ct_r == ct
    if m.sum() > 0:
        ax.scatter(x_r[m], y_r[m], c=CELL_COLORS.get(ct, '#aaa'), s=1.5, alpha=0.8, rasterized=True)

# GC contour
gc_mask = np.isin(ct_r, ['GC_DZ', 'GC_LZ'])
if gc_mask.sum() > 50:
    for c in get_contours(x_r[gc_mask], y_r[gc_mask], sz, threshold=0.2):
        ax.plot(c[:, 1], c[:, 0], color='#2a2a2a', linewidth=1.8, linestyle='--')
    ax.text(np.mean(x_r[gc_mask]), np.mean(y_r[gc_mask]), 'GC',
            fontsize=10, fontweight='bold', color='#2a2a2a', ha='center')

# T zone contour
t_mask = np.isin(ct_r, ['T_naive', 'T_helper', 'T_DN'])
if t_mask.sum() > 100:
    for c in get_contours(x_r[t_mask], y_r[t_mask], sz, threshold=0.3):
        ax.plot(c[:, 1], c[:, 0], color='#2a2a2a', linewidth=1.8, linestyle=':')
    ax.text(np.mean(x_r[t_mask]), np.mean(y_r[t_mask]), 'T zone',
            fontsize=10, fontweight='bold', color='#2a2a2a', ha='center')

ax.set_xlim(0, sz); ax.set_ylim(sz, 0); ax.set_aspect('equal')
ax.set_title('All Cell Types', fontsize=13, fontweight='bold')
ax.set_xticks([]); ax.set_yticks([])
ax.plot([50, 250], [sz-35, sz-35], 'k-', lw=2)
ax.text(150, sz-60, '200 \u03bcm', ha='center', fontsize=9)
for s in ax.spines.values(): s.set_visible(True); s.set_linewidth(1)

# Panels 2-4: B, T, Stromal
for ax, group, title in zip(axes[1:], [b_cells, t_cells, stromal_myeloid],
                             ['B Cells', 'T Cells', 'Stromal & Myeloid']):
    ax.set_facecolor('white')
    sorted_g = sorted([c for c in group if region_counts.get(c, 0) > 0],
                      key=lambda c: -region_counts.get(c, 0))
    for ct in sorted_g:
        m = ct_r == ct
        if m.sum() > 0:
            ax.scatter(x_r[m], y_r[m], c=CELL_COLORS.get(ct), s=2, alpha=0.85, rasterized=True)
    handles = [mpatches.Patch(color=CELL_COLORS[ct],
               label=f'{display_name(ct)} ({region_counts.get(ct,0):,})')
               for ct in sorted_g]
    ax.legend(handles=handles, loc='upper right', fontsize=7, framealpha=0.95)
    ax.set_xlim(0, sz); ax.set_ylim(sz, 0); ax.set_aspect('equal')
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xticks([]); ax.set_yticks([])
    ax.plot([50, 250], [sz-35, sz-35], 'k-', lw=2)
    ax.text(150, sz-60, '200 \u03bcm', ha='center', fontsize=9)
    for s in ax.spines.values(): s.set_visible(True); s.set_linewidth(1)

plt.suptitle('Lymph Node Spatial Organization (1 mm \u00d7 1 mm region)',
             fontsize=15, fontweight='bold', y=1.0)
plt.tight_layout()
plt.show()

---
## 2. LR Database & Spatial Graph

In [ ]:
# Load consensus LR database (LIANA)
lr_df, lr_pairs_all = cst.load_lr_resource('consensus')
print(f'Total LR pairs in database: {len(lr_pairs_all):,}')

# Filter to genes present in Xenium panel
panel_genes = set(adata.var_names)
lr_pairs = [(l, r) for l, r in lr_pairs_all if l in panel_genes and r in panel_genes]
print(f'Testable LR pairs (both genes in panel): {len(lr_pairs):,}')

In [ ]:
# Build k-nearest neighbor spatial graph
k = 10
coords = adata.obsm['spatial']
tree = cKDTree(coords)
distances, indices = tree.query(coords, k=k+1)
indices = indices[:, 1:]    # remove self
distances = distances[:, 1:]

print(f'Spatial graph: {adata.n_obs:,} cells, k={k} neighbors')
print(f'Mean neighbor distance: {distances.mean():.2f} μm')
print(f'Median NN1 distance: {np.median(distances[:, 0]):.2f} μm')

---
## 3. Input Validation & Testable Burden

In [ ]:
# Validate all inputs before running analysis
cst.validate_inputs(adata, cell_type_col, lr_pairs, indices, distances, tau=5.0)

In [ ]:
# Scan cell-type pairs to understand testable burden
scan = cst.scan_celltype_pairs(adata, cell_type_col, lr_pairs, indices, distances)
print(f'\nRecommended tau: {scan["recommended_tau"]} μm')
print(f'Total testing burden: {scan["burden_total"]:,}')

---
## 4. Run CONSTELLATION Analysis

CONSTELLATION tests each (cell-type pair, LR pair) combination using an **analytical permutation test**:

1. **Test statistic** $T = \mathbf{L}^\top \mathbf{w}$: for each sender cell, its ligand expression $L_i$ is weighted by the kernel-weighted receptor signal $w_i = \sum_j K(d_{ij}) R_j$ from receiver neighbors.

2. **Null distribution**: Under random permutation of cell labels (preserving spatial structure), the exact moments are:
   - $E[T] = \frac{\sum L_i \cdot \sum w_i}{n}$
   - $\text{Var}[T] = \frac{\text{SS}_L \cdot \text{SS}_w}{n-1}$ where $\text{SS}_X = \sum (X_i - \bar{X})^2$

3. **z-score**: $(T_{\text{obs}} - E[T]) / \sqrt{\text{Var}[T]}$, with p-value from normal approximation.

4. **FDR correction**: Benjamini-Hochberg across all tests.

In [ ]:
import time

t0 = time.time()
results = cst.run_celltype_analysis(
    adata, cell_type_col,
    lr_pairs=lr_pairs,
    indices=indices, distances=distances,
    tau=5.0, verbose=True
)
elapsed = time.time() - t0
print(f'\nCompleted in {elapsed:.1f}s')
print(f'Total tests: {len(results):,}')

---
## 5. Results Overview

In [ ]:
# Significant interactions: spatially enriched (z > 0) and FDR-corrected (p_adj < 0.05)
sig = results[(results['p_adj'] < 0.05) & (results['z_score'] > 0)]
print(f'Significant interactions: {len(sig):,} / {len(results):,}')

# Top 20 by p_adj
cols = ['sender', 'receiver', 'ligand', 'receptor', 'fold_enrichment', 'z_score', 'p_adj']
sig[cols].head(20)

In [ ]:
# Summary by cell-type pair
pair_summary = sig.groupby(['sender', 'receiver']).agg(
    n_sig=('ligand', 'count'),
    mean_z=('z_score', 'mean'),
    mean_fe=('fold_enrichment', 'mean'),
).reset_index().sort_values('n_sig', ascending=False)

print(f'Cell-type pairs with significant interactions: {len(pair_summary)}')
pair_summary.head(15)

### Cell-type pair visualizations

In [ ]:
# Combined heatmap: within-type on diagonal, between-type off-diagonal
fig, ax = cst.plot_combined_heatmap(
    results, value_col='z_score', agg_func='count',
    cluster=True, title='Significant LR Interactions per Cell-Type Pair'
)
plt.show()

In [ ]:
# Interaction dot plot: size = count, color = mean z-score
fig, ax = cst.plot_interaction_dotplot(
    results, color_col='z_score', cluster=True,
    annotate_threshold=50
)
plt.show()

In [ ]:
# Interaction network: directed edges with arrow width proportional to interaction count
fig, ax = cst.plot_interaction_network(
    results, edge_metric='count', show_self_loops=True,
    min_count=5
)
plt.show()

---
## 6. Biological Validation: Tfh → GC B Cell Interactions

The germinal center (GC) reaction depends on T follicular helper (Tfh) cells providing survival and selection signals to GC B cells. Key literature-known interactions include:

| Ligand–Receptor | Function | Reference |
|:---|:---|:---|
| CD40LG–CD40 | Contact-dependent T cell help | Elgueta 2009 |
| CXCL13–CXCR5 | Chemotaxis/GC homing | Cyster 2019 |
| BTLA–TNFRSF14 | Checkpoint (HVEM) | Murphy 2006 |
| SEMA4D–PLXNB2 | GC organization | Kumanogoh 2005 |
| FCER2–CR2 | B cell activation (sCD23) | Aubry 1992 |
| CD58–CD2 | Adhesion | Davis 2003 |
| CD70–CD27 | Co-stimulation | Borst 2005 |

In [ ]:
# Check known Tfh → GC B cell interactions
validation_pairs = [
    ('Tfh', 'GC_DZ', 'CD40LG', 'CD40', 'T-B help (contact)'),
    ('Tfh', 'GC_LZ', 'CD40LG', 'CD40', 'T-B help (contact)'),
    ('Tfh', 'GC_DZ', 'CXCL13', 'CXCR5', 'Chemokine (secreted)'),
    ('FDC', 'GC_LZ', 'CXCL13', 'CXCR5', 'GC organization'),
    ('Tfh', 'GC_DZ', 'BTLA', 'TNFRSF14', 'Checkpoint'),
    ('Tfh', 'GC_DZ', 'SEMA4D', 'PLXNB2', 'GC organization'),
    ('Tfh', 'GC_DZ', 'FCER2', 'CR2', 'sCD23 (intermediate range)'),
]

print('Biological validation:')
print('-' * 90)
for sender, receiver, lig, rec, biology in validation_pairs:
    match = results[
        (results['sender'] == sender) & (results['receiver'] == receiver) &
        (results['ligand'] == lig) & (results['receptor'] == rec)
    ]
    if len(match) > 0:
        row = match.iloc[0]
        status = 'SIG' if row['p_adj'] < 0.05 and row['z_score'] > 0 else 'ns'
        print(f'  [{status:>3}] {sender} → {receiver}: {lig}–{rec}  '
              f'FE={row["fold_enrichment"]:.2f}  z={row["z_score"]:.2f}  '
              f'p_adj={row["p_adj"]:.2e}  ({biology})')
    else:
        print(f'  [---] {sender} → {receiver}: {lig}–{rec}  NOT TESTED  ({biology})')

In [ ]:
# All significant Tfh → GC_DZ interactions
tfh_gc = sig[(sig['sender'] == 'Tfh') & (sig['receiver'] == 'GC_DZ')].copy()
print(f'Tfh → GC_DZ: {len(tfh_gc)} significant LR pairs')

# Known biology labels
known_biology = {
    ('CD40LG', 'CD40'): 'T-B help',
    ('CXCL13', 'CXCR5'): 'GC chemokine',
    ('BTLA', 'TNFRSF14'): 'Checkpoint',
    ('SEMA4D', 'PLXNB2'): 'GC organization',
    ('FCER2', 'CR2'): 'sCD23 signaling',
    ('CD58', 'CD2'): 'Adhesion',
    ('CD70', 'CD27'): 'Co-stimulation',
}

fig, ax = plt.subplots(figsize=(8, max(4, len(tfh_gc) * 0.35)))
tfh_gc_sorted = tfh_gc.sort_values('fold_enrichment', ascending=True)
labels = [f"{r['ligand']}–{r['receptor']}" for _, r in tfh_gc_sorted.iterrows()]
fe_vals = tfh_gc_sorted['fold_enrichment'].values

# Color bars by whether they are literature-known
colors = ['#5a7aaa' if (r['ligand'], r['receptor']) in known_biology else '#b0b0b0'
          for _, r in tfh_gc_sorted.iterrows()]

bars = ax.barh(range(len(labels)), fe_vals, color=colors, edgecolor='white', linewidth=0.5)

# Annotate known pairs
for i, (_, r) in enumerate(tfh_gc_sorted.iterrows()):
    key = (r['ligand'], r['receptor'])
    if key in known_biology:
        ax.text(fe_vals[i] + 0.02, i, known_biology[key], va='center', fontsize=8,
                fontstyle='italic', color='#333333')

ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels, fontsize=9)
ax.axvline(1.0, color='gray', linestyle='--', linewidth=0.8)
ax.set_xlabel('Fold Enrichment', fontsize=11)
ax.set_title('Tfh → GC DZ: Significant LR Interactions', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 7. LR Dot Plot & Spatial Interaction Plots

In [ ]:
# CellPhoneDB-style LR dot plot — focused on GC biology
gc_ct_pairs = [
    ('Tfh', 'GC_DZ'), ('Tfh', 'GC_LZ'),
    ('FDC', 'GC_DZ'), ('FDC', 'GC_LZ'),
    ('Macrophage', 'T_CD8'), ('Macrophage', 'NK'),
]

fig, ax = cst.plot_lr_dotplot(
    results,
    celltype_pairs=gc_ct_pairs,
    top_n_lr=20,
    cluster_rows=True,
)
plt.show()

In [ ]:
# Spatial interaction plot: Tfh → GC_LZ, CD40LG-CD40 (cell_type mode, full tissue)
fig, ax = cst.plot_spatial_interactions(
    adata, cell_type_col,
    sender='Tfh', receiver='GC_LZ',
    ligand='CD40LG', receptor='CD40',
    indices=indices, distances=distances,
    mode='cell_type', point_size=0.3,
)
plt.show()

In [ ]:
# Zoomed view into GC region
fig, ax = cst.plot_spatial_interactions(
    adata, cell_type_col,
    sender='Tfh', receiver='GC_LZ',
    ligand='CD40LG', receptor='CD40',
    indices=indices, distances=distances,
    mode='cell_type', point_size=2,
    xlim=(4000, 5000), ylim=(5400, 6400),
)
plt.show()

In [ ]:
# Interaction score mode: per-cell CD40LG→CD40 signal strength
fig, ax = cst.plot_spatial_interactions(
    adata, cell_type_col,
    sender='Tfh', receiver='GC_LZ',
    ligand='CD40LG', receptor='CD40',
    indices=indices, distances=distances, tau=5.0,
    mode='score', point_size=0.5,
)
plt.show()

In [ ]:
# Overview mode: spatial distribution of all significant interactions
fig, ax = cst.plot_spatial_interactions(
    adata, cell_type_col,
    results_df=results,
    mode='overview', point_size=0.3,
)
plt.show()

---
## 8. Spatial Range Characterization ($k \times \tau$ Sweep)

CONSTELLATION's kernel decay parameter **$\tau$** controls the effective range of the spatial test, while **$k$** controls the neighborhood size. By sweeping both parameters, we can:

1. **Classify interactions by spatial range** — contact-dependent (CD40LG–CD40) vs secreted (CXCL13–CXCR5)
2. **Assess robustness** — truly enriched interactions are stable across $k$
3. **Compare against other methods** — CellPhoneDB and SpatialDM lack spatial range sensitivity

Pre-computed results from a $6k \times 7\tau$ grid sweep (42 parameter combinations) are loaded below.

In [ ]:
# Load pre-computed grid sweep and benchmark results
sweep_df = pd.read_csv('../results/figure4/grid_sweep_results.csv')
bench_df = pd.read_csv('../results/figure4/benchmark_native_results.csv')

# Unify fold column name
if 'fold' in sweep_df.columns and 'fold_enrichment' in sweep_df.columns:
    sweep_df['fe'] = sweep_df['fold'].fillna(sweep_df['fold_enrichment'])
elif 'fold' in sweep_df.columns:
    sweep_df['fe'] = sweep_df['fold']
else:
    sweep_df['fe'] = sweep_df['fold_enrichment']

print(f'Grid sweep: {len(sweep_df):,} rows')
print(f'  k values: {sorted(sweep_df["k"].unique())}')
print(f'  τ values: {sorted(sweep_df["tau"].unique())}')
print(f'  Cell-type pairs: {sweep_df.groupby(["sender","receiver"]).ngroups}')
print(f'\nBenchmark: {len(bench_df):,} rows')
print(f'  Methods: {sorted(bench_df["method"].unique())}')

In [ ]:
# k × τ grid plots: FE vs τ (lines per k) and FE vs k (lines per τ)
K_VALUES = sorted(sweep_df['k'].unique())
TAU_VALUES = sorted(sweep_df['tau'].unique())

PAIRS = [
    ('CXCL13', 'CXCR5', 'Tfh', 'GC_DZ', 'Secreted'),
    ('CD40LG', 'CD40',  'Tfh', 'GC_LZ', 'Contact'),
]

# Gray gradient: light (small k/τ) → dark (large k/τ)
k_grays = [f'#{v:02x}{v:02x}{v:02x}'
           for v in np.linspace(170, 30, len(K_VALUES)).astype(int)]
k_dashes = ['-', '--', '-.', ':', (0, (3, 1, 1, 1)), (0, (5, 2))]

tau_grays = [f'#{v:02x}{v:02x}{v:02x}'
             for v in np.linspace(170, 30, len(TAU_VALUES)).astype(int)]
tau_dashes = ['-', '--', '-.', ':', (0, (3, 1, 1, 1)), (0, (5, 2)), (0, (1, 1))]

fig, axes = plt.subplots(1, 4, figsize=(16, 3.5), gridspec_kw={'wspace': 0.35})

# Left 2 panels: FE vs τ (lines per k)
for col, (lig, rec, sender, receiver, ltype) in enumerate(PAIRS):
    ax = axes[col]
    pdata = sweep_df[(sweep_df['sender'] == sender) & (sweep_df['receiver'] == receiver) &
                     (sweep_df['ligand'] == lig) & (sweep_df['receptor'] == rec)]
    for ki, k in enumerate(K_VALUES):
        ksub = pdata[pdata['k'] == k].sort_values('tau')
        if len(ksub) == 0:
            continue
        ax.plot(ksub['tau'].values, ksub['fe'].values,
                linestyle=k_dashes[ki], color=k_grays[ki], linewidth=1.8,
                marker='o', markersize=4, label=f'k={k}')
    ax.set_xscale('log')
    ax.set_xticks(TAU_VALUES)
    ax.set_xticklabels(TAU_VALUES, fontsize=8)
    ax.set_xlabel('τ (μm)', fontsize=10)
    ax.set_ylabel('Fold enrichment', fontsize=10)
    ax.set_title(f'{lig}–{rec} ({sender}→{receiver})', fontsize=10, fontweight='bold')
    ax.tick_params(labelsize=8)
    ax.legend(fontsize=7, ncol=2, loc='best', framealpha=0.9)
    all_fe = pdata['fe'].dropna().values
    if len(all_fe) > 0:
        ymin = all_fe.min() - 0.1 * (all_fe.max() - all_fe.min() + 0.01)
        ymax = all_fe.max() + 0.1 * (all_fe.max() - all_fe.min() + 0.01)
        ax.set_ylim(ymin, ymax)

# Right 2 panels: FE vs k (lines per τ)
for col_offset, (lig, rec, sender, receiver, ltype) in enumerate(PAIRS):
    ax = axes[2 + col_offset]
    pdata = sweep_df[(sweep_df['sender'] == sender) & (sweep_df['receiver'] == receiver) &
                     (sweep_df['ligand'] == lig) & (sweep_df['receptor'] == rec)]
    for ti, tau in enumerate(TAU_VALUES):
        tsub = pdata[pdata['tau'] == tau].sort_values('k')
        if len(tsub) == 0:
            continue
        ax.plot(tsub['k'].values, tsub['fe'].values,
                linestyle=tau_dashes[ti], color=tau_grays[ti], linewidth=1.8,
                marker='o', markersize=4, label=f'τ={tau}')
    ax.set_xscale('log')
    ax.set_xticks(K_VALUES)
    ax.set_xticklabels(K_VALUES, fontsize=8)
    ax.set_xlabel('k (neighbors)', fontsize=10)
    ax.set_ylabel('Fold enrichment', fontsize=10)
    ax.set_title(f'{lig}–{rec} ({sender}→{receiver})', fontsize=10, fontweight='bold')
    ax.tick_params(labelsize=8)
    ax.legend(fontsize=7, ncol=2, loc='best', framealpha=0.9)
    all_fe = pdata['fe'].dropna().values
    if len(all_fe) > 0:
        ymin = all_fe.min() - 0.1 * (all_fe.max() - all_fe.min() + 0.01)
        ymax = all_fe.max() + 0.1 * (all_fe.max() - all_fe.min() + 0.01)
        ax.set_ylim(ymin, ymax)

plt.tight_layout()
plt.show()

print('\nInterpretation:')
print('  CXCL13–CXCR5: FE stable across τ and k — secreted chemokine detected at all spatial scales')
print('  CD40LG–CD40: FE peaks at small τ, decays — contact-dependent, signal diluted at long range')

In [ ]:
# Method comparison: CONSTELLATION vs CellPhoneDB vs SpatialDM
# Native metric vs distance, centered to d=15μm reference
METHODS = ['CONSTELLATION', 'CellPhoneDB', 'SpatialDM']
METHOD_COLORS = {
    'CONSTELLATION': '#5a7aaa',
    'CellPhoneDB': '#aa8a5a',
    'SpatialDM': '#b07a7a',
}
CENTERED_LABELS = {
    'CONSTELLATION': '\u0394Fold enrichment',
    'CellPhoneDB': '\u0394L\u00d7R expression',
    'SpatialDM': "Moran's I",
}
LR_CONTEXTS = [
    ('CXCL13', 'CXCR5', 'Tfh', 'GC_DZ', 'Secreted'),
    ('CD40LG', 'CD40',  'Tfh', 'GC_LZ', 'Contact'),
    ('FCER2',  'CR2',   'Tfh', 'GC_DZ', 'Intermediate'),
    ('CXCL12', 'ITGA4', 'Myeloid', 'Treg', 'Secreted'),
]
DIST_VALUES = [6, 15, 30, 60, 150, 300]
REF_DIST = 15

fig, axes = plt.subplots(3, 4, figsize=(16, 9))

for row, method in enumerate(METHODS):
    for col, (lig, rec, sender, receiver, ltype) in enumerate(LR_CONTEXTS):
        ax = axes[row, col]
        sub = bench_df[(bench_df['ligand'] == lig) & (bench_df['receptor'] == rec) &
                       (bench_df['method'] == method) &
                       (bench_df['sender'] == sender) & (bench_df['receiver'] == receiver)]
        sub = sub.sort_values('range_param')

        if len(sub) == 0:
            ax.text(0.5, 0.5, 'No data', transform=ax.transAxes, ha='center',
                    fontsize=8, color='gray')
        else:
            color = METHOD_COLORS[method]
            vals = sub['native_metric'].values.copy()
            # Center CONSTELLATION and CellPhoneDB to reference at d=15μm
            if method in ('CONSTELLATION', 'CellPhoneDB'):
                ref_row = sub[sub['range_param'] == REF_DIST]
                if len(ref_row) > 0:
                    vals = vals - ref_row['native_metric'].values[0]
            ax.plot(sub['range_param'].values, vals,
                    '-o', color=color, linewidth=2, markersize=5, zorder=3)
            ax.fill_between(sub['range_param'].values, 0, vals,
                            alpha=0.15, color=color, zorder=2)
            ax.axhline(0, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)

        ax.set_xscale('log')
        ax.set_xlim(4, 500)
        ax.set_xticks(DIST_VALUES)
        ax.set_xticklabels(DIST_VALUES, fontsize=8)
        ax.minorticks_off()
        if col == 0:
            ax.set_ylabel(CENTERED_LABELS[method], fontsize=9)
        if row == 0:
            ctx = f'{sender}\u2192{receiver}'
            ax.set_title(f'{lig}\u2013{rec}\n({ltype.lower()}, {ctx})',
                         fontsize=9, fontweight='bold')
        if col == 3:
            ax.text(1.08, 0.5, method, transform=ax.transAxes, fontsize=9,
                    fontweight='bold', color=METHOD_COLORS[method], rotation=-90,
                    ha='center', va='center')
        if row == 2:
            ax.set_xlabel('Distance (\u03bcm)', fontsize=9)
        ax.tick_params(labelsize=8)

plt.tight_layout(w_pad=2.0, h_pad=1.5)
plt.subplots_adjust(right=0.93)
plt.show()

print('\nMethod comparison:')
print('  CONSTELLATION: FE captures spatial scale — contact pairs decay, secreted stay flat')
print('  CellPhoneDB: L\u00d7R product decays similarly for both — no spatial scale distinction')
print("  SpatialDM: Moran's I near zero, insensitive to spatial range")

---
## 9. Save Results

In [ ]:
# Save full and significant results
results.to_csv('../results/results_lymphnode_full.csv', index=False)
sig.to_csv('../results/results_lymphnode_significant.csv', index=False)
print(f'Saved: ../results/results_lymphnode_full.csv ({len(results):,} rows)')
print(f'Saved: ../results/results_lymphnode_significant.csv ({len(sig):,} rows)')

---
## Summary

In this notebook we:

1. **Loaded** a 708K-cell human lymph node Xenium dataset with hierarchical cell type annotations
2. **Built** a k=10 nearest-neighbor spatial graph
3. **Ran CONSTELLATION** with the analytical permutation null across all testable combinations
4. **Validated** against known Tfh–GC B cell biology (CD40LG–CD40, CXCL13–CXCR5, etc.)
5. **Visualized** results with combined heatmaps, dot plots, network diagrams, LR dot plots, and spatial overlays
6. **Characterized spatial range** via $k \times \tau$ grid sweep, showing that CONSTELLATION distinguishes contact-dependent (CD40LG–CD40) from secreted (CXCL13–CXCR5) interactions — a capability absent in CellPhoneDB and SpatialDM

### Key advantages of CONSTELLATION
- **Conditions on** spatial structure rather than filtering by it — detects LR interactions even when cell types are not globally colocalized
- **Analytical null** provides exact moments under permutation, with normal approximation via CLT (no Monte Carlo required)
- **Kernel-weighted** neighborhoods give smooth, interpretable distance sensitivity
- **$\tau$ parameter** enables principled classification of interactions by spatial range